# Wide-Angle Camera Model Calibration

This notebook calibrates `data/test_datasets/wide_angle_test.json` with four camera models:

- `OpenCV`
- `PinholeSplined`
- `StereographicOpenCV`
- `StereographicSplined`

It then shows the usual calibration diagnostics and even/odd cross-validation projection-difference plots.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import lensboy as lb
import lensboy.analysis as lba

plt.rcParams["figure.dpi"] = 120

## Load the Dataset

In [ ]:
DATA_PATH = Path("../data/test_datasets/wide_angle_test.json")

# Set to an integer for quick iteration while editing the notebook.
MAX_FRAMES: int | None = None


def load_wide_angle_dataset(path: Path) -> tuple[np.ndarray, list[lb.Frame], int, int]:
    """Load the wide-angle JSON dataset into lensboy target points and frames."""
    data = json.loads(path.read_text())
    image_width = int(data["imageDimensions"]["width"])
    image_height = int(data["imageDimensions"]["height"])

    id_to_index: dict[str, int] = {}
    target_points = []
    for idx, point in enumerate(data["targetPoints"]):
        id_to_index[point["id"]] = idx
        p = point["positionMm"]
        target_points.append([p["x"], p["y"], p["z"]])
    target_points_arr = np.asarray(target_points, dtype=np.float64)

    frames: list[lb.Frame] = []
    samples = data["samples"]
    if MAX_FRAMES is not None:
        samples = samples[:MAX_FRAMES]

    skipped = 0
    for sample in samples:
        indices = []
        pixels = []
        for detection in sample["detections"]:
            point_idx = id_to_index.get(detection["id"])
            if point_idx is None:
                skipped += 1
                continue
            px = detection["pixel"]
            indices.append(point_idx)
            pixels.append([px["x"], px["y"]])
        frames.append(
            lb.Frame(
                target_point_indices=np.asarray(indices, dtype=np.int32),
                detected_points_in_image=np.asarray(pixels, dtype=np.float64),
            )
        )

    if skipped:
        print(f"Skipped {skipped} detections with unknown target point IDs")
    return target_points_arr, frames, image_height, image_width


target_points, frames, image_height, image_width = load_wide_angle_dataset(DATA_PATH)
print(f"target_points: {target_points.shape}")
print(f"frames: {len(frames)}")
print(f"detections: {sum(len(frame) for frame in frames):,}")
print(f"image: {image_width} x {image_height}")

## Helpers

In [ ]:
def summarize_result(name: str, result: lb.CalibrationResult) -> None:
    """Print compact calibration quality numbers."""
    outlier_pct = result.num_outliers() / result.num_detections() * 100
    solved = sum(diag is not None for diag in result.frame_diagnostics)
    print(name)
    print("-" * len(name))
    print(result.camera_model)
    print(f"solved frames: {solved}/{len(result.frames)}")
    print(f"residual sigma MAP: {result.residual_sigma_map():.4f}px")
    print(f"outliers: {result.num_outliers():,}/{result.num_detections():,} ({outlier_pct:.2f}%)")
    if result.target_warp is not None:
        print(f"target warp: {result.target_warp}")


def show_standard_plots(name: str, result: lb.CalibrationResult) -> None:
    """Show the standard calibration diagnostic plots for a result."""
    result.plot_detection_coverage(title=f"{name}: detection coverage")
    result.plot_inlier_coverage(title=f"{name}: inlier coverage")
    result.plot_residuals(title=f"{name}: residuals")
    result.plot_residual_vectors(title=f"{name}: residual vectors")
    result.plot_residual_grid(title=f"{name}: residual grid")
    result.plot_per_image_rms(title=f"{name}: per-image residual RMS")
    result.plot_target_and_poses(title=f"{name}: target and camera poses")

    if result.target_warp is not None:
        result.plot_target_warp(title=f"{name}: target warp")

    try:
        result.plot_distortion_grid(title=f"{name}: distortion grid")  # type: ignore[call-arg]
    except TypeError:
        try:
            result.plot_distortion_grid()
        except Exception as exc:
            print(f"Skipping distortion grid for {name}: {exc}")
    except Exception as exc:
        print(f"Skipping distortion grid for {name}: {exc}")


def calibrate_even_odd(config: lb.CameraModelConfig) -> tuple[lb.CalibrationResult, lb.CalibrationResult]:
    """Fit the same model on even and odd frames for cross-validation."""
    result_even = lb.calibrate_camera(target_points, frames[0::2], camera_model_config=config)
    result_odd = lb.calibrate_camera(target_points, frames[1::2], camera_model_config=config)
    return result_even, result_odd

## OpenCV

In [ ]:
opencv_config = lb.OpenCVConfig(
    image_height=image_height,
    image_width=image_width,
    included_distortion_coefficients=lb.OpenCVConfig.FULL_14,
)

opencv_result = lb.calibrate_camera(
    target_points,
    frames,
    camera_model_config=opencv_config,
)

summarize_result("OpenCV FULL_14", opencv_result)

In [ ]:
show_standard_plots("OpenCV FULL_14", opencv_result)

### OpenCV Cross-Validation

In [ ]:
opencv_even, opencv_odd = calibrate_even_odd(opencv_config)
summarize_result("OpenCV even frames", opencv_even)
print()
summarize_result("OpenCV odd frames", opencv_odd)

lba.plot_projection_diff(
    opencv_even.camera_model,
    opencv_odd.camera_model,
    show_grid=True,
    heatmap_max=1.0
)

## Pinhole Splined

In [ ]:
pinhole_spline_config = lb.PinholeSplinedConfig(
    image_height=image_height,
    image_width=image_width,
    num_knots_x=10,
    num_knots_y=10,
    smoothness_lambda=1.0,
)

pinhole_spline_result = lb.calibrate_camera(
    target_points,
    frames,
    camera_model_config=pinhole_spline_config,
)

summarize_result("Pinhole splined", pinhole_spline_result)

In [ ]:
show_standard_plots("Pinhole splined", pinhole_spline_result)

### Pinhole Splined Cross-Validation

In [ ]:
pinhole_spline_even, pinhole_spline_odd = calibrate_even_odd(pinhole_spline_config)
summarize_result("Pinhole splined even frames", pinhole_spline_even)
print()
summarize_result("Pinhole splined odd frames", pinhole_spline_odd)



In [ ]:

lba.plot_projection_diff(
    pinhole_spline_even.camera_model,
    pinhole_spline_odd.camera_model,
    show_grid=True,
    heatmap_max=1.0
)

## Stereographic OpenCV

In [ ]:
stereo_opencv_config = lb.StereographicOpenCVConfig(
    image_height=image_height,
    image_width=image_width,
    included_distortion_coefficients=lb.StereographicOpenCVConfig.FULL_14,
)

stereo_opencv_result = lb.calibrate_camera(
    target_points,
    frames,
    camera_model_config=stereo_opencv_config,
)

summarize_result("Stereographic OpenCV", stereo_opencv_result)

In [ ]:
show_standard_plots("Stereographic OpenCV", stereo_opencv_result)

### Stereographic OpenCV Cross-Validation

In [ ]:
stereo_opencv_even, stereo_opencv_odd = calibrate_even_odd(stereo_opencv_config)
summarize_result("Stereographic OpenCV even frames", stereo_opencv_even)
print()
summarize_result("Stereographic OpenCV odd frames", stereo_opencv_odd)


In [ ]:

lba.plot_projection_diff(
    stereo_opencv_even.camera_model,
    stereo_opencv_odd.camera_model,
    show_grid=True,
    heatmap_max=1.0
)

## Stereographic Splined

In [ ]:
spline_config = lb.StereographicSplinedConfig(
    image_height=image_height,
    image_width=image_width,
    num_knots_x=10,
    num_knots_y=10,
    smoothness_lambda=1.0,
)

spline_result = lb.calibrate_camera(
    target_points,
    frames,
    camera_model_config=spline_config,
)

summarize_result("Stereographic Splined", spline_result)

In [ ]:
show_standard_plots("Stereographic splined", spline_result)

### Stereographic Splined Cross-Validation

In [ ]:
spline_even, spline_odd = calibrate_even_odd(spline_config)
summarize_result("Stereographic splined even frames", spline_even)
print()
summarize_result("Stereographic splined odd frames", spline_odd)


In [ ]:
lba.plot_projection_diff(
    spline_even.camera_model,
    spline_odd.camera_model,
    show_grid=True,
    heatmap_max=1.0
)

## Full-Data Model Difference

In [ ]:
full_data_comparisons = [
    ("OpenCV vs pinhole splined", opencv_result, pinhole_spline_result),
    ("OpenCV vs stereographic OpenCV", opencv_result, stereo_opencv_result),
    ("Pinhole splined vs stereographic splined", pinhole_spline_result, spline_result),
    ("Stereographic OpenCV vs stereographic splined", stereo_opencv_result, spline_result),
]

for title, result_a, result_b in full_data_comparisons:
    print(title)
    lba.plot_projection_diff(
        result_a.camera_model,
        result_b.camera_model,
        show_grid=True,
        heatmap_max=1.0
    )
    plt.show()